# Solveit login
> Sign in through a browser and save a Solveit session for gateway clients

In [ ]:
#| default_exp login

In [ ]:
#| export
import httpx, secrets, time, webbrowser
from fastcore.utils import *
from fastcore.xdg import xdg_config_home
from fastcore.script import call_parse

Run `solveit-login` and finish signing in in the browser. The command saves the signed `_solveit` cookie value to `$XDG_CONFIG_HOME/solveit/session` (usually `~/.config/solveit/session`), with mode `0600`. An unsuccessful login leaves the previous file unchanged. `--server` selects another Solveit login server, such as a test deployment.

The session grants the same instance access as your browser login. Keep the file private. A copied session is valid until it expires; logging out in the browser does not reliably revoke copies.

In [ ]:
#| export
@call_parse
def main(
    server:str='https://solve.it.com', # Solveit login server
    output:Path=None, # Session file; defaults to $XDG_CONFIG_HOME/solveit/session
):
    "Sign in to Solveit and save a session credential for gateway clients."
    output = output.expanduser() if output else xdg_config_home()/'solveit'/'session'
    paircode = secrets.token_urlsafe(32)
    with httpx.Client(base_url=server.rstrip('/'), headers={'Accept': 'application/json'}) as client:
        r = client.post('/cli_login', data={'paircode': paircode})
        r.raise_for_status()
        print(f'Finish signing in at {r.text}')
        webbrowser.open(r.text)
        deadline = time.monotonic() + 600
        while time.monotonic() < deadline:
            r = client.post('/cli_token', data={'paircode': paircode})
            r.raise_for_status()
            if r.text: break
            time.sleep(1)
        else: raise TimeoutError('Solveit login expired; run solveit-login again')
        token = client.cookies.get('_solveit')
        if not token: raise RuntimeError('Solveit did not return a session cookie')
    with atomic_save(output, mode='w') as f: f.write(token)
    print(f'Saved Solveit session to {output}')

Use the file as a credential source in `~/.config/clikernel/gateways.toml`. Copy the instance URL from your browser and append `/rusty`:

```toml
[gateways.solveit]
url = "https://<instance-host>/rusty"
token_file = "~/.config/solveit/session"
```

Select this named gateway with `create(host="solveit", kernel="py")`. Named gateways do not receive your local environment or working directory. Restart the clikernel client after signing in again.